#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze

In [0]:
df=spark.table("workspace.bronze.erp_px_cat_g1v2")

#Data Transformation

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))

##Data Normalization

In [0]:
df=df.withColumn(
"maintenance",
F.when(F.upper(col("maintenance")) =="YES",F.lit(True))
.when(F.upper(col("maintenance")) =="NO",F.lit(False))
.otherwise(None)
)


##Renaming Columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name,new_name in RENAME_MAP.items():
    df=df.withColumnRenamed(old_name,new_name)

#Sanity check for Dataframe

In [0]:
df.limit(10).display()

#Writing Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_product_category")

#Sanity check for Silve Table

In [0]:
%sql
SELECT * FROM silver.erp_product_category
LIMIT 10;